In [1]:
import pandas as pd
import json

## 1. Load CSV with correct separator

In [2]:
df = pd.read_csv('./data-engineering-test/resources/orders.csv', sep=';')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: './data-engineering-test/resources/orders.csv'

## 2. Rename columns to snake_case

In [ ]:
df = df.rename(columns={'salesowners': 'sales_owners'})
print(f"Columns after rename: {df.columns.tolist()}")

Columns after rename: ['order_id', 'date', 'company_id', 'company_name', 'crate_type', 'contact_data', 'sales_owners']


## 3. Standardise date column to DD/MM/YYYY

In [ ]:
df['date'] = pd.to_datetime(df['date'], format='%d.%m.%y').dt.strftime('%d/%m/%Y')
df['date'].head()

0    29/01/2022
1    21/02/2022
2    03/04/2022
3    14/07/2021
4    23/10/2022
Name: date, dtype: object

## 4. Parse contact_data JSON strings

In [ ]:
def parse_contact_data(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    s = str(val).strip()
    if s.startswith('{'):
        s = '[' + s + ']'
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        if s.count('[') > s.count(']'):
            s = s + ']'
        if s.count('{') > s.count('}'):
            s = s + '}'
        try:
            return json.loads(s)
        except json.JSONDecodeError as e:
            print(f"Failed to parse: {val!r} -> {e}")
            return []

df['contact_data'] = df['contact_data'].apply(parse_contact_data)
df['contact_data'].head(10)

0    [{'contact_name': 'Curtis', 'contact_surname':...
1    [{'contact_name': 'Maria', 'contact_surname': ...
2    [{'contact_name': 'Para', 'contact_surname': '...
3                                                   []
4                                                   []
5    [{'contact_name': 'John', 'contact_surname': '...
6                                                   []
7    [{'contact_name': 'Jennifer', 'contact_surname...
8    [{'contact_name': 'Liav', 'contact_surname': '...
9    [{'contact_name': 'Curtis', 'contact_surname':...
Name: contact_data, dtype: object

## 5. Parse sales_owners into lists

In [ ]:
df['sales_owners'] = df['sales_owners'].apply(
    lambda x: [name.strip() for name in str(x).split(',') if name.strip()] if pd.notna(x) else []
)
df['sales_owners'].head(10)

0      [Leonard Cohen, Luke Skywalker, Ammy Winehouse]
1          [Luke Skywalker, David Goliat, Leon Leonov]
2                                     [Luke Skywalker]
3                        [David Goliat, Leonard Cohen]
4    [Chris Pratt, David Henderson, Marianov Mersch...
5                     [Leonard Cohen, David Henderson]
6                       [Markus Söder, Ammy Winehouse]
7        [David Henderson, Leonard Cohen, Leon Leonov]
8        [Yuri Gagarin, David Goliat, David Henderson]
9           [Ammy Winehouse, Marie Curie, Chris Pratt]
Name: sales_owners, dtype: object

## 6. Fill empty contact_data from matching company_id

In [ ]:
contact_lookup = {}
for _, row in df.iterrows():
    cid = row['company_id']
    if cid not in contact_lookup and len(row['contact_data']) > 0:
        contact_lookup[cid] = row['contact_data']

empty_before = sum(len(c) == 0 for c in df['contact_data'])

def fill_contact(row):
    if len(row['contact_data']) == 0 and row['company_id'] in contact_lookup:
        return contact_lookup[row['company_id']]
    return row['contact_data']

df['contact_data'] = df.apply(fill_contact, axis=1)
empty_after = sum(len(c) == 0 for c in df['contact_data'])
print(f"Empty contact_data: {empty_before} -> {empty_after}")
print(f"\nStill empty (no matching company_id with contact data):")
df[df['contact_data'].apply(len) == 0][['order_id', 'company_id', 'company_name']]

Empty contact_data: 17 -> 14

Still empty (no matching company_id with contact data):


,order_id,company_id,company_name
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,34538e39-cd2e-4641-8d24-3c94146e6f16,Meat Packers Ltd
6,f47ac10b-58cc-4372-a567-0e02b2c3d485,2e90f2b1-d237-47a6-96e8-6d01c0d78c3e,Seafood Supplier GmbH
14,f47ac10b-58cc-4372-a567-0e02b2c3d493,9b31b19f-69a2-4aeb-8f6e-f4b8d2f9c12a,Veggies Unlimited
24,f47ac10b-58cc-4372-a567-0e02b2c3d503,012f20c6-00d5-4f45-999f-12e7639db623,Green World Ltd
29,f47ac10b-58cc-4372-a567-0e02b2c3d508,8347edb3-c4e5-4fc1-8e0c-1fcf9f0d02b9,Organic Foods Ltd
35,f47ac10b-58cc-4372-a567-0e02b2c3d514,0b8755d4-3d28-4039-b9a7-b30cb5ff02ea,Seafood Network Ltd
40,f47ac10b-58cc-4372-a567-0e02b2c3d519,1b21f4a3-22d2-43f2-ae3f-e254d282d9e0,Fresh Veg Co
44,f47ac10b-58cc-4372-a567-0e02b2c3d523,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier
45,f47ac10b-58cc-4372-a567-0e02b2c3d524,34538e39-cd2e-4641-8d24-3c94146e6f16,Meat Packers Ltd


## 7. Detect & fix company name inconsistencies
Only fix where the SAME company_id has different name spellings.

In [ ]:
name_variants = df.groupby('company_id')['company_name'].unique().reset_index()
name_variants.columns = ['company_id', 'name_variants']
name_variants['count'] = name_variants['name_variants'].apply(len)
dupes = name_variants[name_variants['count'] > 1]
print(f"Company IDs with inconsistent names: {len(dupes)}")
for _, row in dupes.iterrows():
    print(f"  {row['company_id']} -> {list(row['name_variants'])}")

# Auto-fix: pick the most frequent name per company_id
company_name_fixes = {}
for _, row in dupes.iterrows():
    cid = row['company_id']
    subset = df[df['company_id'] == cid]
    canonical = subset['company_name'].value_counts().index[0]
    for variant in row['name_variants']:
        if variant != canonical:
            company_name_fixes[variant] = canonical

print(f"\nFixes applied:")
for old, new in company_name_fixes.items():
    print(f"  '{old}' -> '{new}'")

df['company_name'] = df['company_name'].replace(company_name_fixes)
print(f"\nUnique companies after fix: {df['company_name'].nunique()}")

Company IDs with inconsistent names: 2
  1e2b47e6-499e-41c6-91d3-09d12dddfbbd -> ['Fresh Fruits Co', 'Fresh Fruits c.o']
  20dfef10-8f4e-45a1-82fc-123f4ab2a4a5 -> ['healthy snacks c.o.', 'Healthy Snacks Co']

Fixes applied:
  'Fresh Fruits c.o' -> 'Fresh Fruits Co'
  'Healthy Snacks Co' -> 'healthy snacks c.o.'

Unique companies after fix: 40


## 8. Clean contact_data into a single contact_name column
- Combine first + last name into one `contact_name` (e.g. `Curtis Jackson`)
- Set empty values to `Unknown`
- Add `has_contact` boolean

In [ ]:
def extract_contact_name(contacts):
    if contacts and len(contacts) > 0:
        first = contacts[0].get('contact_name', '')
        last = contacts[0].get('contact_surname', '')
        full = f"{first} {last}".strip()
        return full if full else 'Unknown'
    return 'Unknown'

df['has_contact'] = df['contact_data'].apply(lambda x: len(x) > 0)
df['contact_name'] = df['contact_data'].apply(extract_contact_name)

# Drop the raw contact_data column
df = df.drop(columns=['contact_data'])

print(f"has_contact distribution:\n{df['has_contact'].value_counts()}")
print(f"\nSample:")
df[['company_name', 'contact_name', 'has_contact']].head(10)

has_contact distribution:
has_contact
True     48
False    14
Name: count, dtype: int64

Sample:


,company_name,contact_name,has_contact
0,Fresh Fruits Co,Curtis Jackson,True
1,Veggies Inc,Maria Theresa,True
2,Fresh Fruits Co,Para Cetamol,True
3,Seafood Supplier,Unknown,False
4,Meat Packers Ltd,Unknown,False
5,Green Veg Co,John Krasinski,True
6,Seafood Supplier GmbH,Unknown,False
7,Organic Farms,Jennifer Lopez,True
8,Tropical Fruits Ltd,Liav Ichenbaum,True
9,Healthy Snacks,Curtis Jackson,True


## 9. Prepare sales_owners for dashboard
Two dataframes:
- `df_orders` = one row per order (62 rows), sales_owners as comma-separated string + count
- `df_sales` = one row per sales_owner per order (exploded), for salesperson-level analysis

In [ ]:
# df_orders: one row per order
df['sales_owner_count'] = df['sales_owners'].apply(len)
df['sales_owners_str'] = df['sales_owners'].apply(lambda x: ', '.join(x))

df_orders = df.drop(columns=['sales_owners']).copy()

print(f"df_orders shape: {df_orders.shape}")
print(f"Columns: {df_orders.columns.tolist()}")
df_orders.head()

df_orders shape: (62, 9)
Columns: ['order_id', 'date', 'company_id', 'company_name', 'crate_type', 'has_contact', 'contact_name', 'sales_owner_count', 'sales_owners_str']


,order_id,date,company_id,company_name,crate_type,has_contact,contact_name,sales_owner_count,sales_owners_str
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,True,Curtis Jackson,3,"Leonard Cohen, Luke Skywalker, Ammy Winehouse"
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,True,Maria Theresa,3,"Luke Skywalker, David Goliat, Leon Leonov"
2,f47ac10b-58cc-4372-a567-0e02b2c3d481,03/04/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Metal,True,Para Cetamol,1,Luke Skywalker
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,14/07/2021,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier,Plastic,False,Unknown,2,"David Goliat, Leonard Cohen"
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,23/10/2022,34538e39-cd2e-4641-8d24-3c94146e6f16,Meat Packers Ltd,Plastic,False,Unknown,4,"Chris Pratt, David Henderson, Marianov Merschi..."


In [ ]:
# df_sales: one row per sales_owner per order (exploded)
df_sales = df.explode('sales_owners').rename(columns={'sales_owners': 'sales_owner'})
df_sales = df_sales.drop(columns=['sales_owners_str'])

print(f"df_sales shape: {df_sales.shape}")
print(f"Columns: {df_sales.columns.tolist()}")
print(f"\nUnique sales owners: {df_sales['sales_owner'].nunique()}")
print(f"\nOrders per salesperson:")
print(df_sales['sales_owner'].value_counts().to_string())
df_sales.head(10)

df_sales shape: (153, 9)
Columns: ['order_id', 'date', 'company_id', 'company_name', 'crate_type', 'sales_owner', 'has_contact', 'contact_name', 'sales_owner_count']

Unique sales owners: 12

Orders per salesperson:
sales_owner
Leonard Cohen        26
Ammy Winehouse       23
Chris Pratt          18
Leon Leonov          16
David Henderson      16
David Goliat         14
Yuri Gagarin         12
Luke Skywalker       10
Marie Curie           9
Marianov Merschik     5
Markus Söder          2
Vladimir Chukov       2


,order_id,date,company_id,company_name,crate_type,sales_owner,has_contact,contact_name,sales_owner_count
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,Leonard Cohen,True,Curtis Jackson,3
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,Luke Skywalker,True,Curtis Jackson,3
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,Ammy Winehouse,True,Curtis Jackson,3
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,Luke Skywalker,True,Maria Theresa,3
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,David Goliat,True,Maria Theresa,3
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,Leon Leonov,True,Maria Theresa,3
2,f47ac10b-58cc-4372-a567-0e02b2c3d481,03/04/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Metal,Luke Skywalker,True,Para Cetamol,1
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,14/07/2021,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier,Plastic,David Goliat,False,Unknown,2
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,14/07/2021,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier,Plastic,Leonard Cohen,False,Unknown,2
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,23/10/2022,34538e39-cd2e-4641-8d24-3c94146e6f16,Meat Packers Ltd,Plastic,Chris Pratt,False,Unknown,4


## 10. Final validation

In [ ]:
print("=" * 50)
print("df_orders (one row per order)")
print("=" * 50)
print(f"Shape: {df_orders.shape}")
print(f"Columns: {df_orders.columns.tolist()}")
print(f"Unique order_ids: {df_orders['order_id'].nunique()}")
print(f"Unique companies: {df_orders['company_name'].nunique()}")
print(f"Date sample: {df_orders['date'].head(3).tolist()}")
print(f"Unknown contacts: {(df_orders['contact_name'] == 'Unknown').sum()}")

print(f"\n{'=' * 50}")
print("df_sales (one row per sales_owner per order)")
print("=" * 50)
print(f"Shape: {df_sales.shape}")
print(f"Unique order_ids: {df_sales['order_id'].nunique()}")
print(f"Unique sales owners: {df_sales['sales_owner'].nunique()}")

print(f"\n{'=' * 50}")
print("df_orders sample")
print("=" * 50)
df_orders.head(5)

df_orders (one row per order)
Shape: (62, 9)
Columns: ['order_id', 'date', 'company_id', 'company_name', 'crate_type', 'has_contact', 'contact_name', 'sales_owner_count', 'sales_owners_str']
Unique order_ids: 62
Unique companies: 40
Date sample: ['29/01/2022', '21/02/2022', '03/04/2022']
Unknown contacts: 14

df_sales (one row per sales_owner per order)
Shape: (153, 9)
Unique order_ids: 62
Unique sales owners: 12

df_orders sample


,order_id,date,company_id,company_name,crate_type,has_contact,contact_name,sales_owner_count,sales_owners_str
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,True,Curtis Jackson,3,"Leonard Cohen, Luke Skywalker, Ammy Winehouse"
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,True,Maria Theresa,3,"Luke Skywalker, David Goliat, Leon Leonov"
2,f47ac10b-58cc-4372-a567-0e02b2c3d481,03/04/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Metal,True,Para Cetamol,1,Luke Skywalker
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,14/07/2021,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier,Plastic,False,Unknown,2,"David Goliat, Leonard Cohen"
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,23/10/2022,34538e39-cd2e-4641-8d24-3c94146e6f16,Meat Packers Ltd,Plastic,False,Unknown,4,"Chris Pratt, David Henderson, Marianov Merschi..."


In [ ]:
print("df_sales sample")
df_sales.head(10)

df_sales sample


,order_id,date,company_id,company_name,crate_type,sales_owner,has_contact,contact_name,sales_owner_count
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,Leonard Cohen,True,Curtis Jackson,3
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,Luke Skywalker,True,Curtis Jackson,3
0,f47ac10b-58cc-4372-a567-0e02b2c3d479,29/01/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Plastic,Ammy Winehouse,True,Curtis Jackson,3
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,Luke Skywalker,True,Maria Theresa,3
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,David Goliat,True,Maria Theresa,3
1,f47ac10b-58cc-4372-a567-0e02b2c3d480,21/02/2022,0f05a8f1-2bdf-4be7-8c82-4c9b58f04898,Veggies Inc,Wood,Leon Leonov,True,Maria Theresa,3
2,f47ac10b-58cc-4372-a567-0e02b2c3d481,03/04/2022,1e2b47e6-499e-41c6-91d3-09d12dddfbbd,Fresh Fruits Co,Metal,Luke Skywalker,True,Para Cetamol,1
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,14/07/2021,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier,Plastic,David Goliat,False,Unknown,2
3,f47ac10b-58cc-4372-a567-0e02b2c3d482,14/07/2021,1c4b0b50-1d5d-463a-b56e-1a6fd3aeb7d6,Seafood Supplier,Plastic,Leonard Cohen,False,Unknown,2
4,f47ac10b-58cc-4372-a567-0e02b2c3d483,23/10/2022,34538e39-cd2e-4641-8d24-3c94146e6f16,Meat Packers Ltd,Plastic,Chris Pratt,False,Unknown,4
